# Day 5 — Reading data: read vs scan

*100 Days of Polars · Phase 1: Foundations and mental model*

- **Post:** on Medium (link added after publishing)
- **Dataset:** [Olist Brazilian e-commerce](https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce) — CSVs in `data/raw/`
- **Written for:** Polars 2.x (run from the repo root)

In [1]:
import polars as pl

print("polars", pl.__version__)

polars 2.0.0


Every Polars reader comes in two flavours:

| Eager | Lazy | Returns |
|---|---|---|
| `pl.read_csv` | `pl.scan_csv` | `DataFrame` vs `LazyFrame` |
| `pl.read_parquet` | `pl.scan_parquet` | |
| `pl.read_ndjson` | `pl.scan_ndjson` | |

`read_*` loads the data now. `scan_*` only describes *where* the data is; nothing is read until you `collect()`.
In Polars 2.x, `read_csv(...)` is literally `scan_csv(...).collect()` under the hood.

In [2]:
import time
from pathlib import Path

RAW = Path("data/raw")
PROCESSED = Path("data/processed")

df = pl.read_csv(RAW / "olist_orders_dataset.csv")
lf = pl.scan_csv(RAW / "olist_orders_dataset.csv")

print(type(df).__name__, df.shape)
print(type(lf).__name__)

DataFrame (99441, 8)
LazyFrame


In [3]:
# A LazyFrame is a plan, not data
print(lf.explain())

Csv SCAN [data/raw/olist_orders_dataset.csv]
PROJECT */8 COLUMNS
ESTIMATED ROWS: 99185


In [4]:
# Inspect without loading everything
print(lf.collect_schema())
lf.head(3).collect()

Schema({'order_id': String, 'customer_id': String, 'order_status': String, 'order_purchase_timestamp': String, 'order_approved_at': String, 'order_delivered_carrier_date': String, 'order_delivered_customer_date': String, 'order_estimated_delivery_date': String})


order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
str,str,str,str,str,str,str,str
"""e481f51cbdc54678b7cc49136f2d6a…","""9ef432eb6251297304e76186b10a92…","""delivered""","""2017-10-02 10:56:33""","""2017-10-02 11:07:15""","""2017-10-04 19:55:00""","""2017-10-10 21:25:13""","""2017-10-18 00:00:00"""
"""53cdb2fc8bc7dce0b6741e21502734…","""b0830fb4747a6c6d20dea0b8c802d7…","""delivered""","""2018-07-24 20:41:37""","""2018-07-26 03:24:27""","""2018-07-26 14:31:00""","""2018-08-07 15:27:45""","""2018-08-13 00:00:00"""
"""47770eb9100c2d0c44946d9cf07ec6…","""41ce2a54c0b03bf3443c3d931a3670…","""delivered""","""2018-08-08 08:38:49""","""2018-08-08 08:55:23""","""2018-08-08 13:50:00""","""2018-08-17 18:06:29""","""2018-09-04 00:00:00"""


## 1. Reading only what you need

In [5]:
# Pick columns and rows at read time
pl.read_csv(
    RAW / "olist_orders_dataset.csv",
    columns=["order_id", "order_status", "order_purchase_timestamp"],
    n_rows=1_000,
).shape

(1000, 3)

## 2. Getting the types right at read time

Two fixes you will use constantly with real CSVs:

In [6]:
# Zip-code prefixes look like numbers, so they are inferred as integers and lose leading zeros
inferred = pl.read_csv(RAW / "olist_customers_dataset.csv")
print(inferred.schema["customer_zip_code_prefix"], inferred["customer_zip_code_prefix"].head(3).to_list())

fixed = pl.read_csv(
    RAW / "olist_customers_dataset.csv",
    schema_overrides={"customer_zip_code_prefix": pl.String},
)
print(fixed.schema["customer_zip_code_prefix"], fixed["customer_zip_code_prefix"].head(3).to_list())

Int64 [14409, 9790, 1151]
String ['14409', '09790', '01151']


In [7]:
# Timestamps stay strings unless you ask Polars to parse them
plain = pl.read_csv(RAW / "olist_orders_dataset.csv")
parsed = pl.read_csv(RAW / "olist_orders_dataset.csv", try_parse_dates=True)
print(plain.schema["order_purchase_timestamp"], "->", parsed.schema["order_purchase_timestamp"])

String -> Datetime(time_unit='us', time_zone=None)


In [8]:
# The translation file starts with a UTF-8 byte-order mark (BOM). Polars strips it on read:
raw_names = pl.read_csv(RAW / "product_category_name_translation.csv").columns
print([repr(c) for c in raw_names])  # no '\ufeff' in the first name

# with_column_names cleans every name on read, e.g. messy headers like "Order ID " -> "order_id"
clean = pl.read_csv(
    RAW / "product_category_name_translation.csv",
    with_column_names=lambda cols: [c.strip().lower().replace(" ", "_") for c in cols],
)
print(clean.columns)

["'product_category_name'", "'product_category_name_english'"]
['product_category_name', 'product_category_name_english']


> **Polars 1.x → 2.x:** `read_csv` has no `dtypes=` (use `schema_overrides=`), `rechunk=` or `n_threads=` parameters, and `columns=[2, 1, 3]` returns columns in the order you asked for.

## 3. CSV vs Parquet

CSV is text: every read re-parses every number and date. Parquet stores typed, compressed columns, with min/max statistics per row group.

In [9]:
PROCESSED.mkdir(parents=True, exist_ok=True)
orders = pl.read_csv(RAW / "olist_orders_dataset.csv", try_parse_dates=True)
orders.write_parquet(PROCESSED / "orders.parquet")

csv_mb = (RAW / "olist_orders_dataset.csv").stat().st_size / 1e6
pq_mb = (PROCESSED / "orders.parquet").stat().st_size / 1e6
print(f"CSV {csv_mb:.1f} MB -> Parquet {pq_mb:.1f} MB")

CSV 17.7 MB -> Parquet 6.2 MB


In [10]:
def best_of(fn, runs=5):
    times = []
    for _ in range(runs):
        t0 = time.perf_counter()
        fn()
        times.append(time.perf_counter() - t0)
    return min(times)


t_csv = best_of(lambda: pl.read_csv(RAW / "olist_orders_dataset.csv", try_parse_dates=True))
t_pq = best_of(lambda: pl.read_parquet(PROCESSED / "orders.parquet"))
print(f"read_csv {t_csv * 1000:.0f} ms | read_parquet {t_pq * 1000:.0f} ms")

read_csv 134 ms | read_parquet 29 ms


## 4. Where scan pays off

With `scan_*`, a filter and a column selection travel **into the reader**. With Parquet, whole row groups can be skipped using their statistics.

In [11]:
query = (
    pl.scan_parquet(PROCESSED / "orders.parquet")
    .filter(pl.col("order_status") == "canceled")
    .select("order_id", "order_purchase_timestamp")
)
print(query.explain())  # look for the selection and the filter inside the Parquet SCAN

simple π 2/2 ["order_id", ... 1 other column]
  Parquet SCAN [data/processed/orders.parquet]
  PROJECT 3/8 COLUMNS
  SELECTION: col("order_status") == "canceled"
  ESTIMATED ROWS: 99441


In [12]:
t_read = best_of(
    lambda: pl.read_parquet(PROCESSED / "orders.parquet")
    .filter(pl.col("order_status") == "canceled")
    .select("order_id", "order_purchase_timestamp")
)
t_scan = best_of(lambda: query.collect())
print(f"read then filter {t_read * 1000:.1f} ms | scan with pushdown {t_scan * 1000:.1f} ms")

read then filter 28.2 ms | scan with pushdown 8.8 ms


On a file this small the difference is modest. Polars wrote it as two row groups, but canceled orders fall inside the `order_status` range of both, so neither can be skipped. On the NYC taxi data later in the series, it decides whether a query fits in memory at all.

## Results for the post

| | CSV | Parquet |
|---|---|---|
| File size (MB) | 17.7 | 6.2 |
| Full read (ms) | 134 | 29 |

## Try it yourself

1. Convert all 9 CSVs to Parquet in a loop and compare total sizes.
2. Read `olist_geolocation_dataset.csv` with `infer_schema_length=10`. Does inference still get the types right?